# Addresses26 — BN7 Validation Notebook v3

Purpose: validate the Addresses26 Land Registry BN pipeline against a direct Land Registry postcode download saved as:

`/content/drive/MyDrive/Addresses26_Data/derived/land_registry_price_paid/BN7.csv`

This notebook:
- mounts Google Drive
- loads all yearly `pp_bn_standard_YYYY.parquet` files
- loads and normalises the direct `BN7.csv`
- compares record-level keys
- checks row counts, duplicates, date ranges, prices, and postcode filters
- saves validation outputs to `reports/validation/`

In [1]:
# ============================================================
# 01. SETUP
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import json
import re
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

BASE = Path("/content/drive/MyDrive/Addresses26_Data")
DERIVED = BASE / "derived" / "land_registry_price_paid"
REPORTS = BASE / "reports" / "validation"
REPORTS.mkdir(parents=True, exist_ok=True)

REFERENCE_CSV = DERIVED / "BN7.csv"
POSTCODE_PREFIX = "BN7"

print("BASE     :", BASE)
print("DERIVED  :", DERIVED)
print("REPORTS  :", REPORTS)
print("REFERENCE:", REFERENCE_CSV)

Mounted at /content/drive
BASE     : /content/drive/MyDrive/Addresses26_Data
DERIVED  : /content/drive/MyDrive/Addresses26_Data/derived/land_registry_price_paid
REPORTS  : /content/drive/MyDrive/Addresses26_Data/reports/validation
REFERENCE: /content/drive/MyDrive/Addresses26_Data/derived/land_registry_price_paid/BN7.csv


In [2]:
# ============================================================
# 02. SMALL HELPERS
# ============================================================

def normalise_colname(c):
    return (
        str(c)
        .strip()
        .lower()
        .replace("\ufeff", "")
        .replace(" ", "_")
        .replace("-", "_")
        .replace("/", "_")
        .replace(".", "")
    )

def file_sha256(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def first_existing_column(df, candidates):
    cols = list(df.columns)
    for c in candidates:
        if c in cols:
            return c
    return None

def parse_price_series(s):
    return (
        s.astype(str)
        .str.replace("£", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
        .replace({"": np.nan, "nan": np.nan, "None": np.nan})
        .pipe(pd.to_numeric, errors="coerce")
    )

def parse_date_series(s):
    # Land Registry bulk parquet normally uses ISO YYYY-MM-DD.
    # Direct CSV downloads may use deed_date or date_of_transfer, generally parseable.
    return pd.to_datetime(s, errors="coerce", dayfirst=False)

def clean_postcode_series(s):
    return (
        s.astype(str)
        .str.upper()
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .replace({"NAN": np.nan, "NONE": np.nan, "": np.nan})
    )

def make_match_key(df):
    # Key intentionally uses exact price + date + postcode.
    # This is stronger than price+date alone and avoids false matches.
    return (
        df["price"].round(0).astype("Int64").astype(str)
        + "|"
        + df["date"].dt.strftime("%Y-%m-%d").astype(str)
        + "|"
        + df["postcode"].astype(str)
    )

In [3]:
# ============================================================
# 03. LOAD DIRECT BN7 REFERENCE CSV
# ============================================================

assert REFERENCE_CSV.exists(), f"Reference CSV not found: {REFERENCE_CSV}"

# Try utf-8-sig first, then latin1 fallback.
try:
    bn7_ref_raw = pd.read_csv(REFERENCE_CSV, dtype=str, encoding="utf-8-sig")
except UnicodeDecodeError:
    bn7_ref_raw = pd.read_csv(REFERENCE_CSV, dtype=str, encoding="latin1")

bn7_ref_raw.columns = [normalise_colname(c) for c in bn7_ref_raw.columns]

print("Reference rows raw:", len(bn7_ref_raw))
print("Reference columns:")
print(list(bn7_ref_raw.columns))
display(bn7_ref_raw.head())

Reference rows raw: 11161
Reference columns:
['unique_id', 'price_paid', 'deed_date', 'postcode', 'property_type', 'new_build', 'estate_type', 'saon', 'paon', 'street', 'locality', 'town', 'district', 'county', 'transaction_category', 'linked_data_uri']


,unique_id,price_paid,deed_date,postcode,property_type,new_build,estate_type,saon,paon,street,locality,town,district,county,transaction_category,linked_data_uri
0,4638A9C3-DD2D-421F-8200-EAE4A7B66303,250000,2009-08-19,BN7 1AD,T,N,F,1,CASTLE VIEW,WESTGATE STREET,NaN,LEWES,LEWES,EAST SUSSEX,A,http://landregistry.data.gov.uk/data/ppi/trans...
1,E2A4C681-4F3A-429E-AA6D-87479AF3D55E,250000,2004-08-13,BN7 1AD,D,N,F,1,CASTLE VIEW,WESTGATE STREET,NaN,LEWES,LEWES,EAST SUSSEX,A,http://landregistry.data.gov.uk/data/ppi/trans...
2,43D25782-F8D2-4681-A6A8-B59909168358,165000,2002-06-11,BN7 1AD,D,N,F,1,CASTLE VIEW,WESTGATE STREET,NaN,LEWES,LEWES,EAST SUSSEX,A,http://landregistry.data.gov.uk/data/ppi/trans...
3,00C240D8-A19F-41DB-BA77-4E67464AFE8A,80000,1997-04-11,BN7 1AD,D,N,F,1,CASTLE VIEW,WESTGATE STREET,NaN,LEWES,LEWES,EAST SUSSEX,A,http://landregistry.data.gov.uk/data/ppi/trans...
4,98BA9A1F-CC48-42CC-A2A1-7291EEEE4739,425000,2009-06-18,BN7 1AD,T,N,F,2,CASTLE VIEW,WESTGATE STREET,NaN,LEWES,LEWES,EAST SUSSEX,A,http://landregistry.data.gov.uk/data/ppi/trans...


In [4]:
# ============================================================
# 04. NORMALISE REFERENCE CSV SCHEMA
# ============================================================

# Supports both:
# - bulk/pipeline names: price, date_of_transfer, postcode, ppd_category_type
# - Land Registry app download names: price_paid, deed_date, postcode, ppd_category_type

price_col = first_existing_column(bn7_ref_raw, [
    "price", "price_paid", "amount", "transaction_price"
])

date_col = first_existing_column(bn7_ref_raw, [
    "date_of_transfer", "deed_date", "transfer_date", "date"
])

postcode_col = first_existing_column(bn7_ref_raw, [
    "postcode", "post_code"
])

category_col = first_existing_column(bn7_ref_raw, [
    "ppd_category_type", "ppd_category", "category_type", "record_status"
])

missing_core = []
if price_col is None:
    missing_core.append("price/price_paid")
if date_col is None:
    missing_core.append("date_of_transfer/deed_date")
if postcode_col is None:
    missing_core.append("postcode")

assert not missing_core, (
    "Reference CSV missing required columns after normalisation: "
    + str(missing_core)
    + "\nAvailable columns: "
    + str(list(bn7_ref_raw.columns))
)

bn7_ref = pd.DataFrame({
    "price": parse_price_series(bn7_ref_raw[price_col]),
    "date": parse_date_series(bn7_ref_raw[date_col]),
    "postcode": clean_postcode_series(bn7_ref_raw[postcode_col]),
})

# Optional standard transaction filter if present.
if category_col is not None:
    bn7_ref["ppd_category_type"] = bn7_ref_raw[category_col].astype(str).str.upper().str.strip()
    before = len(bn7_ref)
    bn7_ref = bn7_ref[bn7_ref["ppd_category_type"].eq("A")].copy()
    print(f"Applied reference standard-only filter using {category_col}: {before} -> {len(bn7_ref)}")
else:
    print("No category column found in reference CSV; assuming direct download is acceptable for comparison.")

bn7_ref = bn7_ref[bn7_ref["postcode"].str.startswith(POSTCODE_PREFIX, na=False)].copy()
bn7_ref_clean = bn7_ref[["price", "date", "postcode"]].dropna().copy()
bn7_ref_clean["match_key"] = make_match_key(bn7_ref_clean)

print("Reference rows cleaned:", len(bn7_ref_clean))
print("Reference date range:", bn7_ref_clean["date"].min(), "to", bn7_ref_clean["date"].max())
print("Reference duplicate keys:", bn7_ref_clean["match_key"].duplicated().sum())
display(bn7_ref_clean.head())

No category column found in reference CSV; assuming direct download is acceptable for comparison.
Reference rows cleaned: 11161
Reference date range: 1995-01-03 00:00:00 to 2026-02-25 00:00:00
Reference duplicate keys: 38


,price,date,postcode,match_key
0,250000,2009-08-19,BN7 1AD,250000|2009-08-19|BN7 1AD
1,250000,2004-08-13,BN7 1AD,250000|2004-08-13|BN7 1AD
2,165000,2002-06-11,BN7 1AD,165000|2002-06-11|BN7 1AD
3,80000,1997-04-11,BN7 1AD,80000|1997-04-11|BN7 1AD
4,425000,2009-06-18,BN7 1AD,425000|2009-06-18|BN7 1AD


In [5]:
# ============================================================
# 05. LOAD PIPELINE PARQUET FILES
# ============================================================

parquet_files = sorted(
    p for p in DERIVED.glob("pp_bn_standard_*.parquet")
    if "all_years" not in p.name.lower()
)

assert parquet_files, f"No yearly parquet files found in {DERIVED}"

print("Parquet files found:", len(parquet_files))
for p in parquet_files:
    print("-", p.name)

pipeline_frames = []
load_log = []

for p in parquet_files:
    df = pd.read_parquet(p)
    df.columns = [normalise_colname(c) for c in df.columns]
    df["source_file"] = p.name
    pipeline_frames.append(df)
    load_log.append({"file": p.name, "rows": len(df), "columns": list(df.columns)})

master = pd.concat(pipeline_frames, ignore_index=True)

print("Master pipeline rows:", len(master))
print("Master columns:")
print(list(master.columns))
display(master.head())

Parquet files found: 32
- pp_bn_standard_1995.parquet
- pp_bn_standard_1996.parquet
- pp_bn_standard_1997.parquet
- pp_bn_standard_1998.parquet
- pp_bn_standard_1999.parquet
- pp_bn_standard_2000.parquet
- pp_bn_standard_2001.parquet
- pp_bn_standard_2002.parquet
- pp_bn_standard_2003.parquet
- pp_bn_standard_2004.parquet
- pp_bn_standard_2005.parquet
- pp_bn_standard_2006.parquet
- pp_bn_standard_2007.parquet
- pp_bn_standard_2008.parquet
- pp_bn_standard_2009.parquet
- pp_bn_standard_2010.parquet
- pp_bn_standard_2011.parquet
- pp_bn_standard_2012.parquet
- pp_bn_standard_2013.parquet
- pp_bn_standard_2014.parquet
- pp_bn_standard_2015.parquet
- pp_bn_standard_2016.parquet
- pp_bn_standard_2017.parquet
- pp_bn_standard_2018.parquet
- pp_bn_standard_2019.parquet
- pp_bn_standard_2020.parquet
- pp_bn_standard_2021.parquet
- pp_bn_standard_2022.parquet
- pp_bn_standard_2023.parquet
- pp_bn_standard_2024.parquet
- pp_bn_standard_2025.parquet
- pp_bn_standard_2026.parquet
Master pipeline 

,transaction_id,price,date_of_transfer,postcode,property_type,old_new,duration,paon,saon,street,...,ppd_category_type,record_status,postcode_area,postcode_district,postcode_sector,property_type_label,old_new_label,duration_label,address_compact,source_file
0,{75E42F17-A6E8-4FC5-BA60-576388CAA756},30000,1995-01-02,BN1 6DJ,F,N,L,2,FLAT 7,FLORENCE ROAD,...,A,A,BN,<NA>,None,Flat/Maisonette,Established,Leasehold,"FLAT 7, 2, FLORENCE ROAD, BRIGHTON, BRIGHTON",pp_bn_standard_1995.parquet
1,{CC94C6AF-87D8-4E78-A86B-B7E69B880706},78000,1995-01-03,BN1 3LD,T,N,F,23,<NA>,CROWN GARDENS,...,A,A,BN,<NA>,None,Terraced,Established,Freehold,"23, CROWN GARDENS, BRIGHTON, BRIGHTON",pp_bn_standard_1995.parquet
2,{2A289EA0-0395-CDC8-E050-A8C063054829},72500,1995-01-03,BN1 4AQ,S,N,F,40,<NA>,GLOUCESTER ROAD,...,A,A,BN,<NA>,None,Semi-detached,Established,Freehold,"40, GLOUCESTER ROAD, BRIGHTON",pp_bn_standard_1995.parquet
3,{2E813E5F-7919-4BAB-AA08-9DEB69DFE2AD},51000,1995-01-03,BN1 4EL,T,N,F,34,<NA>,TIDY STREET,...,A,A,BN,<NA>,None,Terraced,Established,Freehold,"34, TIDY STREET, BRIGHTON, BRIGHTON",pp_bn_standard_1995.parquet
4,{C43E2279-3057-4AA6-A752-41D3D6F91FF5},52250,1995-01-03,BN10 7EE,S,N,F,14,<NA>,ST LAURENCE CLOSE,...,A,A,BN,<NA>,None,Semi-detached,Established,Freehold,"14, ST LAURENCE CLOSE, TELSCOMBE CLIFFS, PEACE...",pp_bn_standard_1995.parquet


In [6]:
# ============================================================
# 06. NORMALISE PIPELINE DATA AND EXTRACT BN7
# ============================================================

pipeline_price_col = first_existing_column(master, ["price", "price_paid"])
pipeline_date_col = first_existing_column(master, ["date_of_transfer", "deed_date", "date"])
pipeline_postcode_col = first_existing_column(master, ["postcode", "post_code"])
pipeline_category_col = first_existing_column(master, ["ppd_category_type", "ppd_category", "category_type"])

missing_pipeline = []
if pipeline_price_col is None:
    missing_pipeline.append("price/price_paid")
if pipeline_date_col is None:
    missing_pipeline.append("date_of_transfer/deed_date/date")
if pipeline_postcode_col is None:
    missing_pipeline.append("postcode")

assert not missing_pipeline, (
    "Pipeline parquet missing required columns: "
    + str(missing_pipeline)
    + "\nAvailable columns: "
    + str(list(master.columns))
)

pipe = pd.DataFrame({
    "price": parse_price_series(master[pipeline_price_col]),
    "date": parse_date_series(master[pipeline_date_col]),
    "postcode": clean_postcode_series(master[pipeline_postcode_col]),
})

if pipeline_category_col is not None:
    pipe["ppd_category_type"] = master[pipeline_category_col].astype(str).str.upper().str.strip()

bn7_master = pipe[pipe["postcode"].str.startswith(POSTCODE_PREFIX, na=False)].copy()

# Pipeline files should already be standard-only. This check is only a safeguard.
if "ppd_category_type" in bn7_master.columns:
    non_a = (~bn7_master["ppd_category_type"].eq("A")).sum()
    print("Pipeline non-standard rows in BN7:", int(non_a))

bn7_master_clean = bn7_master[["price", "date", "postcode"]].dropna().copy()
bn7_master_clean["match_key"] = make_match_key(bn7_master_clean)

print("BN7 pipeline rows cleaned:", len(bn7_master_clean))
print("BN7 pipeline date range:", bn7_master_clean["date"].min(), "to", bn7_master_clean["date"].max())
print("BN7 pipeline duplicate keys:", bn7_master_clean["match_key"].duplicated().sum())
display(bn7_master_clean.head())

Pipeline non-standard rows in BN7: 0
BN7 pipeline rows cleaned: 11161
BN7 pipeline date range: 1995-01-03 00:00:00 to 2026-02-25 00:00:00
BN7 pipeline duplicate keys: 38


,price,date,postcode,match_key
29,30000,1995-01-03,BN7 1JY,30000|1995-01-03|BN7 1JY
184,60000,1995-01-06,BN7 2AW,60000|1995-01-06|BN7 2AW
185,150000,1995-01-06,BN7 2BE,150000|1995-01-06|BN7 2BE
375,53500,1995-01-13,BN7 1LH,53500|1995-01-13|BN7 1LH
376,42000,1995-01-13,BN7 2LT,42000|1995-01-13|BN7 2LT


In [7]:
# ============================================================
# 07. RECORD-LEVEL COMPARISON
# ============================================================

ref_keys = set(bn7_ref_clean["match_key"])
pipe_keys = set(bn7_master_clean["match_key"])

matched_keys = ref_keys & pipe_keys
missing_keys = ref_keys - pipe_keys
extra_keys = pipe_keys - ref_keys

matched_records = len(matched_keys)
missing_in_pipeline = bn7_ref_clean[bn7_ref_clean["match_key"].isin(missing_keys)].copy()
extra_in_pipeline = bn7_master_clean[bn7_master_clean["match_key"].isin(extra_keys)].copy()

match_rate = matched_records / len(ref_keys) if ref_keys else np.nan

print("Matched records       :", matched_records)
print("Missing in pipeline   :", len(missing_in_pipeline))
print("Extra in pipeline     :", len(extra_in_pipeline))
print(f"Match rate vs reference: {match_rate:.2%}")

if len(missing_in_pipeline):
    print("\nMissing sample:")
    display(missing_in_pipeline.head(20))

if len(extra_in_pipeline):
    print("\nExtra sample:")
    display(extra_in_pipeline.head(20))

Matched records       : 11123
Missing in pipeline   : 0
Extra in pipeline     : 0
Match rate vs reference: 100.00%


In [8]:
# ============================================================
# 08. QA CHECKS
# ============================================================

qa = {}

qa["created_utc"] = datetime.now(timezone.utc).isoformat()
qa["postcode_prefix"] = POSTCODE_PREFIX
qa["reference_csv"] = str(REFERENCE_CSV)
qa["reference_sha256"] = file_sha256(REFERENCE_CSV)
qa["parquet_files"] = [p.name for p in parquet_files]
qa["reference_rows_clean"] = int(len(bn7_ref_clean))
qa["pipeline_bn7_rows_clean"] = int(len(bn7_master_clean))
qa["matched_records"] = int(matched_records)
qa["missing_in_pipeline"] = int(len(missing_in_pipeline))
qa["extra_in_pipeline"] = int(len(extra_in_pipeline))
qa["match_rate_vs_reference"] = float(match_rate) if not pd.isna(match_rate) else None
qa["reference_duplicate_keys"] = int(bn7_ref_clean["match_key"].duplicated().sum())
qa["pipeline_duplicate_keys"] = int(bn7_master_clean["match_key"].duplicated().sum())
qa["reference_date_min"] = str(bn7_ref_clean["date"].min())
qa["reference_date_max"] = str(bn7_ref_clean["date"].max())
qa["pipeline_date_min"] = str(bn7_master_clean["date"].min())
qa["pipeline_date_max"] = str(bn7_master_clean["date"].max())
qa["reference_price_min"] = float(bn7_ref_clean["price"].min()) if len(bn7_ref_clean) else None
qa["reference_price_max"] = float(bn7_ref_clean["price"].max()) if len(bn7_ref_clean) else None
qa["pipeline_price_min"] = float(bn7_master_clean["price"].min()) if len(bn7_master_clean) else None
qa["pipeline_price_max"] = float(bn7_master_clean["price"].max()) if len(bn7_master_clean) else None
qa["validation_passed"] = bool(
    len(missing_in_pipeline) == 0
    and len(extra_in_pipeline) == 0
    and len(bn7_ref_clean) > 0
)

pd.DataFrame([qa]).T.rename(columns={0: "value"})

,value
created_utc,2026-04-27T12:55:35.578163+00:00
postcode_prefix,BN7
reference_csv,/content/drive/MyDrive/Addresses26_Data/derive...
reference_sha256,0ddc83d634713275526828b2586bfda409176dd8a4b258...
parquet_files,"[pp_bn_standard_1995.parquet, pp_bn_standard_1..."
reference_rows_clean,11161
pipeline_bn7_rows_clean,11161
matched_records,11123
missing_in_pipeline,0
extra_in_pipeline,0


In [9]:
# ============================================================
# 09. SAVE VALIDATION OUTPUTS
# ============================================================

summary_path = REPORTS / "bn7_validation_summary.json"
missing_path = REPORTS / "bn7_missing_in_pipeline.csv"
extra_path = REPORTS / "bn7_extra_in_pipeline.csv"
load_log_path = REPORTS / "bn7_validation_parquet_load_log.json"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(qa, f, indent=2, default=str)

missing_in_pipeline.to_csv(missing_path, index=False)
extra_in_pipeline.to_csv(extra_path, index=False)

with open(load_log_path, "w", encoding="utf-8") as f:
    json.dump(load_log, f, indent=2, default=str)

print("Saved:")
print("-", summary_path)
print("-", missing_path)
print("-", extra_path)
print("-", load_log_path)

if qa["validation_passed"]:
    print("\n✅ VALIDATION PASSED — BN7 direct download matches pipeline output.")
else:
    print("\n⚠️ VALIDATION DIFFERENCES FOUND — inspect missing/extra CSV outputs.")

Saved:
- /content/drive/MyDrive/Addresses26_Data/reports/validation/bn7_validation_summary.json
- /content/drive/MyDrive/Addresses26_Data/reports/validation/bn7_missing_in_pipeline.csv
- /content/drive/MyDrive/Addresses26_Data/reports/validation/bn7_extra_in_pipeline.csv
- /content/drive/MyDrive/Addresses26_Data/reports/validation/bn7_validation_parquet_load_log.json

✅ VALIDATION PASSED — BN7 direct download matches pipeline output.


## Interpretation

A pass means the postcode-level direct Land Registry download agrees with the derived pipeline output for BN7 using exact:

`price + date + postcode`

Small differences can happen if the direct download and the bulk yearly files were downloaded on different dates, because Land Registry data can be revised or late-registered.